# 10 × 5 STV: MGGG Projection vs. Our Simulation

Reproduces the "Projected number of city council seats" comparison from the
original MGGG-style repo's `ensembles/Analysis.ipynb`
(`df10x5[groups].hist(figsize=(40,4), layout=(1,4), bins=list(range(30)))`),
laid out here as the top row of a 3×4 grid, with two additional rows showing
the same seat-count distributions from our own VoteKit-based election
simulation for the **10 X 5 STV - Asian Bloc Separate** run
(`configs/asian-seperate-bloc.json`) -- one row per voter model
(Plackett-Luce, Bradley-Terry).

All three rows share the same four columns (Asian, Black, Hispanic, White)
and the same `bins=range(30)` x-axis, so each column is a direct before/after
comparison of that group's projected vs. simulated seat count.

This notebook depends on a sibling checkout of the original repo at
`/Users/brocksauvage/Documents/Github/chicago` for `projection/projection_10x5.csv`
-- it isn't part of this repo.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
# pipeline/summarize_results.py's functions use bare relative paths (e.g.
# Path("outputs")/..., config["geodata_path"]) that assume cwd is the repo
# root, same as when run.py drives them -- match that here so this notebook
# can call those functions directly instead of re-deriving the same logic.
os.chdir(REPO_ROOT)

from run import load_config
from pipeline.summarize_results import (
    build_summary_dataframe, aggregate_to_plan_level,
    _compute_representation_baselines, _prepare_directories,
)

# Sibling checkout of the original MGGG-style repo -- not part of this repo,
# referenced only for its precomputed projection CSV.
MGGG_REPO_ROOT = Path("/Users/brocksauvage/Documents/Github/chicago")

GROUPS = ["Asian", "Black", "Hispanic", "White"]
BINS = list(range(30))

## Load the MGGG repo's precomputed 10×5 projection

In [ ]:
mggg_10x5 = pd.read_csv(MGGG_REPO_ROOT / "projection" / "projection_10x5.csv")
mggg_10x5[GROUPS].describe()

## Load our own "10 X 5 STV - Asian Bloc Separate" run

In [ ]:
config = load_config(str(REPO_ROOT / "configs" / "asian-seperate-bloc.json"))
run_name = config["run_name"]
i_cs_turnout = _compute_representation_baselines(config)[1]
results_dir, summary_dir, figs_dir = _prepare_directories(run_name)
df = build_summary_dataframe(config, results_dir, i_cs_turnout)
df_plan = aggregate_to_plan_level(df)

# Slate -> group label, matching GROUPS' naming/order so all three rows share
# the same columns.
SLATE_TO_GROUP = {"A": "Asian", "B": "Black", "H": "Hispanic", "W": "White"}

pl_seats = df_plan[df_plan["mode"] == "slate_pl"]
bt_seats = df_plan[df_plan["mode"] == "slate_bt"]
print(f"Plackett-Luce instances: {len(pl_seats)}, Bradley-Terry instances: {len(bt_seats)}")

## 3×4 comparison grid

In [ ]:
BAR_COLOR = "#1780ab"

fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharex=True)

# (row label, data) -- data=None means "read straight from mggg_10x5[group]"
# instead of "seats_<slate>" from one of our own mode-filtered subsets.
ROW_SPECS = [
    ("MGGG Projection\n(chicago/ensembles/Analysis.ipynb)", None),
    ("Our Simulation\nPlackett-Luce", pl_seats),
    ("Our Simulation\nBradley-Terry", bt_seats),
]

for row_idx, (row_label, data) in enumerate(ROW_SPECS):
    for col_idx, group in enumerate(GROUPS):
        ax = axes[row_idx][col_idx]
        if data is None:
            values = mggg_10x5[group]
        else:
            slate = next(s for s, g in SLATE_TO_GROUP.items() if g == group)
            values = data[f"seats_{slate}"]
        ax.hist(values, bins=BINS, color=BAR_COLOR, edgecolor="white", linewidth=0.5)
        if row_idx == 0:
            ax.set_title(group, fontsize=13, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=10)
        if row_idx == len(ROW_SPECS) - 1:
            ax.set_xlabel("Seats Won")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

fig.suptitle(
    "10 × 5 STV: MGGG Projection vs. Our Simulation (Asian Bloc Separate)",
    fontsize=16, fontweight="bold", y=1.02,
)
fig.tight_layout()
plt.show()